# Bayesian Regime Project: Roadmap and EDA

This notebook turns our project plan into a runnable workflow. It has two goals:

1. lock the modeling direction for the final project
2. explore the prepared dataset before fitting any regime model


## 1. Project Direction

**Core question**

Can latent market regimes inferred from macro conditions improve dynamic allocation across diversified ETFs?

**Working dataset**

- Monthly data from `2006-03-31` to `2026-04-30`
- Asset return columns:
  - `SPY_ret`, `EFA_ret`, `TLT_ret`, `GLD_ret`, `DBC_ret`, `VNQ_ret`
- Macro signal columns:
  - `T10Y2Y_z`, `VIX_z`, `CREDIT_z`, `UNRATE_z`, `UNRATE_CHG_z`, `INDPRO_YOY_z`, `CFNAI_z`

**Recommended first model**

We will start with a Bayesian Hidden Markov Model using the 6 asset return series as observations.

- latent state `z_t`: market regime at month `t`
- candidate numbers of regimes: `K = 2` and `K = 3`
- regime transitions: first-order Markov chain with persistence
- observation model: regime-specific multivariate Gaussian returns
- macro variables: used mainly to interpret and label the inferred regimes

**Why this is a good first version**

- It directly models the asset returns needed for portfolio construction.
- It is much easier to fit than a full 13-variable joint HMM.
- It still lets us explain each regime economically using macro indicators.

**Planned deliverables**

1. exploratory analysis of the dataset
2. baseline regime model
3. Bayesian HMM fit for `K=2` and `K=3`
4. regime interpretation with macro overlays
5. portfolio allocation rule based on posterior regime probabilities
6. backtest and comparison against simple baselines


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

ROOT = Path.cwd()
DATA_PATH = ROOT / "macro_asset_dataset.csv"

RETURN_COLS = ["SPY_ret", "EFA_ret", "TLT_ret", "GLD_ret", "DBC_ret", "VNQ_ret"]
MACRO_COLS = [
    "T10Y2Y_z",
    "VIX_z",
    "CREDIT_z",
    "UNRATE_z",
    "UNRATE_CHG_z",
    "INDPRO_YOY_z",
    "CFNAI_z",
]


## 2. Load the Dataset

In [ ]:
df = pd.read_csv(DATA_PATH, index_col=0, parse_dates=True)
df.index.name = "date"

print("Shape:", df.shape)
print("Date range:", df.index.min().date(), "to", df.index.max().date())
print("Columns:")
for col in df.columns:
    print("-", col)

df.head()

## 3. Data Quality Check

In [ ]:
missing = df.isna().sum().to_frame(name="missing_count")
missing

In [ ]:
summary_stats = df.describe().T[["mean", "std", "min", "max"]]
summary_stats.round(4)

## 4. Asset Return Behavior

A good regime model should have a reason to exist. We want to see whether asset behavior changes over time and whether risky assets tend to move together.

In [ ]:
cumulative = (1 + df[RETURN_COLS]).cumprod()

ax = cumulative.plot(figsize=(12, 6), linewidth=2)
ax.set_title("Cumulative Growth of $1 Across Asset ETFs")
ax.set_xlabel("Date")
ax.set_ylabel("Value")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[RETURN_COLS].corr(), annot=True, cmap="RdBu_r", center=0)
plt.title("Return Correlation Matrix")
plt.show()

In [ ]:
rolling_vol = df[RETURN_COLS].rolling(12).std() * np.sqrt(12)

ax = rolling_vol.plot(figsize=(12, 6), linewidth=1.8)
ax.set_title("12-Month Rolling Annualized Volatility")
ax.set_xlabel("Date")
ax.set_ylabel("Volatility")
ax.grid(alpha=0.3)
plt.show()

## 5. Macro Signal Behavior

The macro variables are already standardized, so values above and below zero are easier to compare visually.

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[MACRO_COLS].corr(), annot=True, cmap="RdBu_r", center=0)
plt.title("Macro Signal Correlation Matrix")
plt.show()

In [ ]:
fig, axes = plt.subplots(len(MACRO_COLS), 1, figsize=(12, 16), sharex=True)

for ax, col in zip(axes, MACRO_COLS):
    ax.plot(df.index, df[col], linewidth=1.6)
    ax.axhline(0, color="black", linewidth=0.8, alpha=0.5)
    ax.set_title(col)
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

## 6. Quick Correlation Checks

This gives us a compact way to see which variables move together most strongly.

In [ ]:
corr = df.corr()

return_pairs = []
for i, a in enumerate(RETURN_COLS):
    for b in RETURN_COLS[i + 1:]:
        return_pairs.append((a, b, corr.loc[a, b]))

return_corrs = (
    pd.DataFrame(return_pairs, columns=["asset_1", "asset_2", "correlation"])
    .assign(abs_correlation=lambda x: x["correlation"].abs())
    .sort_values("abs_correlation", ascending=False)
)

return_corrs.head(10)

In [ ]:
return_macro_pairs = []
for a in RETURN_COLS:
    for b in MACRO_COLS:
        return_macro_pairs.append((a, b, corr.loc[a, b]))

return_macro_corrs = (
    pd.DataFrame(return_macro_pairs, columns=["asset", "macro", "correlation"])
    .assign(abs_correlation=lambda x: x["correlation"].abs())
    .sort_values("abs_correlation", ascending=False)
)

return_macro_corrs.head(12)

## 7. Main Takeaways So Far

The figures and summary tables support the idea that this dataset is appropriate for regime modeling.

- The cumulative return chart shows that `SPY` and `GLD` were the strongest long-run performers in this sample, with cumulative returns of about `534.8%` and `421.0%`, while `DBC` was much weaker at about `14.5%`.
- The return correlation heatmap shows a clear risk block: `SPY` and `EFA` have correlation `0.872`, `SPY` and `VNQ` have correlation `0.752`, and `EFA` and `VNQ` have correlation `0.725`.
- The rolling volatility chart shows that market risk is not constant over time, which is exactly the kind of behavior a regime model is meant to capture.
- The return-macro correlation table shows only modest one-to-one relationships. The strongest absolute example is `TLT_ret` with `T10Y2Y_z` at about `0.127`, which suggests that a latent-state approach may be more informative than a simple single-factor forecasting rule.

Overall, the exploratory analysis points to two useful ideas for the project: asset returns move in clusters, and the market environment appears to change over time rather than stay in one stable state.


## 8. What We Should Do Next

The next step should be to rebuild the macro standardization more carefully for backtesting.

Right now, the macro `z`-scores were built using the full sample, which creates mild look-ahead bias. A safer version is:

- expanding-window standardization, or
- training-period standardization applied to later periods

After that, we can build a first baseline regime model and then the Bayesian HMM.


## 9. Rebuild Macro Standardization Without Full-Sample Look-Ahead

The current modeling table is convenient, but the macro `z`-scores were created using the full sample. That means each early observation was scaled using information from the future.

A safer approach is to rebuild the macro scaling with an expanding window, so each date is standardized only using information available up to that point in the sample.


In [ ]:
RAW_DATA_PATH = ROOT / "macro_asset_raw.csv"
SAFE_DATA_PATH = ROOT / "macro_asset_dataset_expanding_z.csv"

raw_df = pd.read_csv(RAW_DATA_PATH, index_col=0, parse_dates=True)
raw_df.index.name = "date"

RAW_MACRO_COLS = ["T10Y2Y", "VIX", "CREDIT", "UNRATE", "UNRATE_CHG", "INDPRO_YOY", "CFNAI"]
SAFE_MACRO_COLS = [f"{col}_z_exp" for col in RAW_MACRO_COLS]

raw_df[RAW_MACRO_COLS].head()

In [ ]:
def expanding_zscore(frame: pd.DataFrame, min_periods: int = 24) -> pd.DataFrame:
    """Standardize each column using only information available up to each date."""
    exp_mean = frame.expanding(min_periods=min_periods).mean()
    exp_std = frame.expanding(min_periods=min_periods).std()
    z = (frame - exp_mean) / exp_std.replace(0, np.nan)
    return z


macro_expanding_z = expanding_zscore(raw_df[RAW_MACRO_COLS], min_periods=24)
macro_expanding_z.columns = SAFE_MACRO_COLS

safe_df = raw_df[RETURN_COLS].join(macro_expanding_z, how="inner").dropna()

print("Backtest-safe dataset shape:", safe_df.shape)
print("Backtest-safe date range:", safe_df.index.min().date(), "to", safe_df.index.max().date())

safe_df.head()

In [ ]:
comparison = pd.DataFrame({
    "full_sample_z": df["VIX_z"],
    "expanding_z": safe_df["VIX_z_exp"],
}).dropna()

ax = comparison.plot(figsize=(12, 5), linewidth=1.8)
ax.set_title("VIX: Full-Sample vs Expanding-Window Standardization")
ax.set_xlabel("Date")
ax.set_ylabel("Standardized Value")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
safe_summary = safe_df.describe().T[["mean", "std", "min", "max"]]
safe_summary.round(4)

In [ ]:
safe_df.to_csv(SAFE_DATA_PATH)
print("Saved:", SAFE_DATA_PATH)


## 10. Updated Takeaway

The expanding-window standardization step gives us a more credible modeling dataset for the rest of the notebook.

- The backtest-safe dataset has `217` monthly observations from `2008-02-29` to `2026-04-30`.
- This shorter sample is the tradeoff for avoiding look-ahead bias in the macro scaling step.
- The key benefit is that every macro `z`-score is now based only on information available up to that point in time.

This means the remaining regime and backtest sections are based on a dataset that is much more defensible for a final project.


## 11. Baseline Regime Model

As a first benchmark, we will fit a **Gaussian mixture model** on the six asset return series.

This is not yet a full Hidden Markov Model because it does not explicitly model regime transitions over time. But it is still useful because it gives us:

- a simple regime classification baseline
- candidate numbers of regimes to compare
- an interpretable first segmentation of market environments

Later, the Bayesian HMM will improve on this by modeling both regime membership and regime persistence jointly.


In [ ]:
from sklearn.mixture import GaussianMixture

X_returns = safe_df[RETURN_COLS].copy()
candidate_regimes = [2, 3]

gmm_results = {}
comparison_rows = []

for k in candidate_regimes:
    gmm = GaussianMixture(
        n_components=k,
        covariance_type="full",
        init_params="random",
        random_state=42,
        n_init=20,
    )
    gmm.fit(X_returns)
    labels = gmm.predict(X_returns)
    probs = gmm.predict_proba(X_returns)

    gmm_results[k] = {
        "model": gmm,
        "labels": labels,
        "probs": probs,
    }
    comparison_rows.append({
        "K": k,
        "AIC": gmm.aic(X_returns),
        "BIC": gmm.bic(X_returns),
        "log_likelihood": gmm.score(X_returns) * len(X_returns),
    })

model_comparison = pd.DataFrame(comparison_rows).sort_values("BIC").reset_index(drop=True)
model_comparison

In [ ]:
best_k = int(model_comparison.loc[0, "K"])
best_result = gmm_results[best_k]

print(f"Selected baseline model: K = {best_k} regimes (lowest BIC)")

In [ ]:
baseline_df = safe_df.copy()
baseline_df["raw_regime"] = best_result["labels"]

state_rank = (
    baseline_df.groupby("raw_regime")[RETURN_COLS]
    .mean()
    .mean(axis=1)
    .sort_values()
)
regime_map = {old_state: new_state for new_state, old_state in enumerate(state_rank.index)}

baseline_df["baseline_regime"] = baseline_df["raw_regime"].map(regime_map)

ordered_probs = pd.DataFrame(index=baseline_df.index)
for old_state, new_state in regime_map.items():
    ordered_probs[f"regime_{new_state}_prob"] = best_result["probs"][:, old_state]

baseline_df = pd.concat([baseline_df, ordered_probs], axis=1)
baseline_df.head()

In [ ]:
regime_summary = baseline_df.groupby("baseline_regime")[RETURN_COLS + SAFE_MACRO_COLS].mean()
regime_counts = baseline_df["baseline_regime"].value_counts().sort_index().rename("n_months")
regime_share = (regime_counts / len(baseline_df)).rename("share_of_sample")
regime_eq_return = baseline_df.groupby("baseline_regime")[RETURN_COLS].mean().mean(axis=1).rename("avg_equal_weight_return")

regime_report = pd.concat([regime_counts, regime_share, regime_eq_return], axis=1).join(regime_summary)
regime_report.round(4)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={"height_ratios": [1, 2]})

axes[0].step(baseline_df.index, baseline_df["baseline_regime"], where="mid", linewidth=1.8)
axes[0].set_title(f"Baseline Regime Assignment Over Time (K={best_k})")
axes[0].set_ylabel("Regime")
axes[0].grid(alpha=0.3)

eq_growth = (1 + baseline_df[RETURN_COLS].mean(axis=1)).cumprod()
axes[1].plot(baseline_df.index, eq_growth, linewidth=2, label="Equal-weight growth")
axes[1].set_title("Equal-Weight Portfolio Growth")
axes[1].set_ylabel("Value")
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
prob_cols = [col for col in baseline_df.columns if col.endswith("_prob")]

ax = baseline_df[prob_cols].plot(figsize=(12, 5), linewidth=1.8)
ax.set_title("Baseline Regime Membership Probabilities")
ax.set_xlabel("Date")
ax.set_ylabel("Probability")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
switch_rate = (baseline_df["baseline_regime"] != baseline_df["baseline_regime"].shift(1)).mean()
print(f"Approximate month-to-month regime switching rate: {switch_rate:.3f}")

baseline_output_path = ROOT / "baseline_regime_results.csv"
baseline_df.to_csv(baseline_output_path)
print("Saved:", baseline_output_path)

## 12. How To Read The Baseline Results

The Gaussian mixture model is a useful first-pass regime classifier even though it does not explicitly model persistence.

In the current results:

- the model-selection table prefers `K=2` over `K=3` by BIC
- the lower-return regime covers only `17` months, or about `7.8%` of the sample
- the average equal-weight monthly return in that regime is about `-4.65%`, compared with about `0.90%` in the more common regime
- the stress regime also lines up with worse macro conditions: `VIX_z_exp` is about `1.216`, `CREDIT_z_exp` is about `1.078`, and `UNRATE_z_exp` is about `1.355`
- the approximate switching rate is about `10.6%`, which suggests the classifier is not flipping every month

So the baseline model already finds a small crisis-like cluster, but it still treats each month more independently than we would like. That is why the later Bayesian and Markov-switching models are important.


## 13. Bayesian Regime Model

We will now fit a **Bayesian 2-state Hidden Markov Model** to the equal-weight portfolio return.

This is a practical first Bayesian version for the current environment:

- the hidden state follows a Markov chain, so regime persistence is modeled directly
- each regime has its own mean and variance for the portfolio return
- we use Gibbs sampling with conjugate priors, so we do not need PyMC or NumPyro installed

This is narrower than a full multivariate regime model on all six asset returns, but it already captures the key Bayesian idea: latent regimes with posterior uncertainty over time.


In [ ]:
from scipy.special import logsumexp
from scipy.stats import invgamma, norm

bayes_df = safe_df.copy()
bayes_df["equal_weight_return"] = bayes_df[RETURN_COLS].mean(axis=1)

y = bayes_df["equal_weight_return"].to_numpy()
T = len(y)
K = 2

bayes_df[["equal_weight_return"]].head()

In [ ]:
def sample_states_ffbs(y, pi, transition, mu, sigma2, rng):
    """Forward-filter backward-sample for a Gaussian HMM."""
    T = len(y)
    K = len(mu)
    log_emission = np.column_stack([
        norm.logpdf(y, loc=mu[k], scale=np.sqrt(sigma2[k]))
        for k in range(K)
    ])

    log_alpha = np.zeros((T, K))
    log_alpha[0] = np.log(pi + 1e-12) + log_emission[0]

    for t in range(1, T):
        for k in range(K):
            log_alpha[t, k] = log_emission[t, k] + logsumexp(
                log_alpha[t - 1] + np.log(transition[:, k] + 1e-12)
            )

    states = np.zeros(T, dtype=int)
    p_last = np.exp(log_alpha[-1] - logsumexp(log_alpha[-1]))
    states[-1] = rng.choice(K, p=p_last)

    for t in range(T - 2, -1, -1):
        log_p = log_alpha[t] + np.log(transition[:, states[t + 1]] + 1e-12)
        probs = np.exp(log_p - logsumexp(log_p))
        states[t] = rng.choice(K, p=probs)

    return states


def sample_emission_params(y, states, mu0, kappa0, alpha0, beta0, rng):
    """Sample regime-specific mean and variance from a Normal-Inverse-Gamma posterior."""
    K = len(np.unique(states)) if len(np.unique(states)) > 0 else 2
    mu = np.zeros(2)
    sigma2 = np.zeros(2)

    for k in range(2):
        yk = y[states == k]
        nk = len(yk)

        if nk == 0:
            sigma2[k] = invgamma.rvs(a=alpha0, scale=beta0, random_state=rng)
            mu[k] = rng.normal(mu0, np.sqrt(sigma2[k] / kappa0))
            continue

        ybar = yk.mean()
        sse = np.sum((yk - ybar) ** 2)

        kappa_n = kappa0 + nk
        mu_n = (kappa0 * mu0 + nk * ybar) / kappa_n
        alpha_n = alpha0 + nk / 2
        beta_n = beta0 + 0.5 * sse + 0.5 * (kappa0 * nk / kappa_n) * (ybar - mu0) ** 2

        sigma2[k] = invgamma.rvs(a=alpha_n, scale=beta_n, random_state=rng)
        mu[k] = rng.normal(mu_n, np.sqrt(sigma2[k] / kappa_n))

    return mu, sigma2


def relabel_by_mean(states, mu, sigma2, transition, pi):
    """Order regimes by mean return so state 0 is the lower-return regime."""
    order = np.argsort(mu)
    reverse_map = np.empty_like(order)
    reverse_map[order] = np.arange(len(order))
    new_states = reverse_map[states]
    new_mu = mu[order]
    new_sigma2 = sigma2[order]
    new_transition = transition[np.ix_(order, order)]
    new_pi = pi[order]
    return new_states, new_mu, new_sigma2, new_transition, new_pi


def run_bayesian_hmm(y, init_states, n_iter=1600, burn=600, thin=5, seed=42):
    rng = np.random.default_rng(seed)
    obs_std = np.std(y)

    mu0 = 0.0
    kappa0 = 1.0
    alpha0 = 3.0
    beta0 = (alpha0 - 1) * (obs_std ** 2)

    alpha_pi = np.array([1.0, 1.0])
    alpha_transition = np.array([[12.0, 2.0], [2.0, 12.0]])

    states = init_states.copy().astype(int)
    pi = np.array([0.5, 0.5])
    transition = np.array([[0.9, 0.1], [0.1, 0.9]])
    mu = np.array([np.quantile(y, 0.25), np.quantile(y, 0.75)])
    sigma2 = np.array([obs_std ** 2, obs_std ** 2])

    state_prob_sum = np.zeros((len(y), 2))
    transition_sum = np.zeros((2, 2))
    pi_sum = np.zeros(2)
    mu_draws = []
    sigma2_draws = []
    saved_states = []

    for it in range(n_iter):
        states = sample_states_ffbs(y, pi, transition, mu, sigma2, rng)
        mu, sigma2 = sample_emission_params(y, states, mu0, kappa0, alpha0, beta0, rng)

        transition_counts = np.zeros((2, 2))
        for t in range(len(states) - 1):
            transition_counts[states[t], states[t + 1]] += 1

        for k in range(2):
            transition[k] = rng.dirichlet(alpha_transition[k] + transition_counts[k])

        pi = rng.dirichlet(alpha_pi + np.bincount(states[:1], minlength=2))
        states, mu, sigma2, transition, pi = relabel_by_mean(states, mu, sigma2, transition, pi)

        if it >= burn and (it - burn) % thin == 0:
            state_prob_sum += np.eye(2)[states]
            transition_sum += transition
            pi_sum += pi
            mu_draws.append(mu.copy())
            sigma2_draws.append(sigma2.copy())
            saved_states.append(states.copy())

    n_saved = len(saved_states)
    return {
        "state_probs": state_prob_sum / n_saved,
        "transition_mean": transition_sum / n_saved,
        "pi_mean": pi_sum / n_saved,
        "mu_draws": np.array(mu_draws),
        "sigma2_draws": np.array(sigma2_draws),
        "saved_states": np.array(saved_states),
        "n_saved": n_saved,
    }


In [ ]:
if "baseline_df" in globals() and "baseline_regime" in baseline_df.columns:
    init_states = baseline_df.loc[bayes_df.index, "baseline_regime"].to_numpy()
else:
    init_states = (y < np.median(y)).astype(int)

bayes_hmm_result = run_bayesian_hmm(
    y=y,
    init_states=init_states,
    n_iter=1600,
    burn=600,
    thin=5,
    seed=42,
)

print("Saved posterior draws:", bayes_hmm_result["n_saved"])
print("Posterior mean transition matrix:")
print(np.round(bayes_hmm_result["transition_mean"], 3))

In [ ]:
bayes_df["prob_regime_0"] = bayes_hmm_result["state_probs"][:, 0]
bayes_df["prob_regime_1"] = bayes_hmm_result["state_probs"][:, 1]
bayes_df["bayesian_regime_map"] = bayes_hmm_result["state_probs"].argmax(axis=1)

mu_draws = bayes_hmm_result["mu_draws"]
sigma_draws = np.sqrt(bayes_hmm_result["sigma2_draws"])

posterior_summary = pd.DataFrame({
    "mean_return_posterior_mean": mu_draws.mean(axis=0),
    "mean_return_posterior_sd": mu_draws.std(axis=0),
    "volatility_posterior_mean": sigma_draws.mean(axis=0),
    "volatility_posterior_sd": sigma_draws.std(axis=0),
    "avg_state_probability": bayes_hmm_result["state_probs"].mean(axis=0),
}, index=["regime_0_low_return", "regime_1_high_return"])

posterior_summary.round(4)

In [ ]:
bayesian_regime_report = bayes_df.groupby("bayesian_regime_map")[RETURN_COLS + SAFE_MACRO_COLS + ["equal_weight_return"]].mean()
bayesian_regime_counts = bayes_df["bayesian_regime_map"].value_counts().sort_index().rename("n_months")
bayesian_regime_share = (bayesian_regime_counts / len(bayes_df)).rename("share_of_sample")

pd.concat([bayesian_regime_counts, bayesian_regime_share], axis=1).join(bayesian_regime_report).round(4)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(bayes_df.index, bayes_df["prob_regime_0"], linewidth=1.8, color="tab:red")
axes[0].set_title("Posterior Probability of Low-Return Regime")
axes[0].set_ylabel("Probability")
axes[0].grid(alpha=0.3)

axes[1].plot(bayes_df.index, bayes_df["equal_weight_return"], linewidth=1.5, color="tab:blue")
axes[1].set_title("Equal-Weight Portfolio Return")
axes[1].set_ylabel("Return")
axes[1].grid(alpha=0.3)

axes[2].step(bayes_df.index, bayes_df["bayesian_regime_map"], where="mid", linewidth=1.8, color="tab:green")
axes[2].set_title("Most Likely Bayesian Regime")
axes[2].set_ylabel("Regime")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
eq_growth = (1 + bayes_df["equal_weight_return"]).cumprod()
colors = bayes_df["bayesian_regime_map"].map({0: "tab:red", 1: "tab:green"})

plt.figure(figsize=(12, 5))
plt.scatter(bayes_df.index, eq_growth, c=colors, s=18, alpha=0.85)
plt.plot(bayes_df.index, eq_growth, color="black", linewidth=1, alpha=0.6)
plt.title("Equal-Weight Portfolio Growth Colored by Bayesian Regime")
plt.ylabel("Value")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
bayesian_output_path = ROOT / "bayesian_hmm_results.csv"
bayes_df.to_csv(bayesian_output_path)
print("Saved:", bayesian_output_path)

## 14. Bayesian Model Interpretation

This section is the clearest Bayesian piece of the notebook, and the results are economically sensible.

Key posterior findings:

- the posterior mean monthly return in the low-return regime is about `-1.71%`
- the posterior mean monthly return in the high-return regime is about `0.81%`
- the low-return regime is much more volatile, with posterior mean volatility about `5.83%` versus about `2.70%` in the high-return regime
- the posterior transition matrix is persistent, with approximate diagonal entries `0.825` and `0.952`
- about `17.1%` of the average posterior mass is assigned to the low-return regime, and there are about `25` months where the low-return regime probability is above `0.5`

This is a strong Bayesian proof of concept: the model separates a rarer, more volatile stress regime from a more common, calmer regime and does so with explicit posterior uncertainty.


## 15. Preferred Dynamic Regime Model: Markov-Switching Regression

For the final project, this is the regime model we should probably emphasize.

Why this is a strong choice:

- it models regime persistence directly through transition probabilities
- it is a standard financial time-series regime model
- it fits cleanly in the current environment using `statsmodels`
- it is easier to explain than a custom sampler or a fragile high-dimensional model

We will fit a **2-state Markov-switching regression** on the equal-weight portfolio return, with different means and variances across regimes.


In [ ]:
import warnings

import statsmodels.api as sm

ms_df = safe_df.copy()
ms_df["equal_weight_return"] = ms_df[RETURN_COLS].mean(axis=1)

ms_y = ms_df["equal_weight_return"]
np.random.seed(42)

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="A date index has been provided")
    ms_model = sm.tsa.MarkovRegression(
        ms_y,
        k_regimes=2,
        trend="c",
        switching_variance=True,
    )
    ms_result = ms_model.fit(
        em_iter=10,
        search_reps=20,
        search_iter=10,
        disp=False,
    )

print(ms_result.summary())

In [ ]:
raw_means = np.array([ms_result.params[f"const[{i}]"] for i in range(2)])
raw_vars = np.array([ms_result.params[f"sigma2[{i}]"] for i in range(2)])
raw_transition = ms_result.regime_transition[:, :, 0].T

# Reorder states so regime 0 is always the lower-return regime.
ms_order = np.argsort(raw_means)
ordered_means = raw_means[ms_order]
ordered_vols = np.sqrt(raw_vars[ms_order])
ordered_transition = raw_transition[np.ix_(ms_order, ms_order)]

ms_probs = ms_result.smoothed_marginal_probabilities.iloc[:, ms_order].copy()
ms_probs.columns = [f"ms_regime_{i}_prob" for i in range(2)]

ms_df = pd.concat([ms_df, ms_probs], axis=1)
ms_df["ms_regime"] = ms_probs.to_numpy().argmax(axis=1)

ms_param_summary = pd.DataFrame(
    {
        "mean_return": ordered_means,
        "volatility": ordered_vols,
        "expected_duration_months": 1 / (1 - np.diag(ordered_transition)),
    },
    index=["regime_0_low_return", "regime_1_high_return"],
)

ms_param_summary.round(4)

In [ ]:
ms_regime_report = ms_df.groupby("ms_regime")[RETURN_COLS + SAFE_MACRO_COLS + ["equal_weight_return"]].mean()
ms_regime_counts = ms_df["ms_regime"].value_counts().sort_index().rename("n_months")
ms_regime_share = (ms_regime_counts / len(ms_df)).rename("share_of_sample")

pd.concat([ms_regime_counts, ms_regime_share], axis=1).join(ms_regime_report).round(4)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(ms_df.index, ms_df["ms_regime_0_prob"], linewidth=1.8, color="tab:red")
axes[0].set_title("Smoothed Probability of Low-Return Markov-Switching Regime")
axes[0].set_ylabel("Probability")
axes[0].grid(alpha=0.3)

axes[1].plot(ms_df.index, ms_df["equal_weight_return"], linewidth=1.5, color="tab:blue")
axes[1].set_title("Equal-Weight Portfolio Return")
axes[1].set_ylabel("Return")
axes[1].grid(alpha=0.3)

axes[2].step(ms_df.index, ms_df["ms_regime"], where="mid", linewidth=1.8, color="tab:green")
axes[2].set_title("Most Likely Markov-Switching Regime")
axes[2].set_ylabel("Regime")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
eq_growth_ms = (1 + ms_df["equal_weight_return"]).cumprod()
colors_ms = ms_df["ms_regime"].map({0: "tab:red", 1: "tab:green"})

plt.scatter(ms_df.index, eq_growth_ms, c=colors_ms, s=18, alpha=0.85)
plt.plot(ms_df.index, eq_growth_ms, color="black", linewidth=1, alpha=0.6)
plt.title("Equal-Weight Portfolio Growth Colored by Markov-Switching Regime")
plt.ylabel("Value")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
markov_switching_output_path = ROOT / "markov_switching_results.csv"
ms_df.to_csv(markov_switching_output_path)
print("Saved:", markov_switching_output_path)

## 16. Why This Should Be The Main Regime Model

The Markov-switching regression is the most stable and presentation-ready dynamic regime model in the notebook.

The fitted two-state model produces a very intuitive interpretation:

- the low-return regime has mean monthly return about `-1.84%` and volatility about `6.68%`
- the high-return regime has mean monthly return about `0.78%` and volatility about `2.58%`
- expected regime durations are about `5.15` months for the low-return state and `38.73` months for the high-return state
- the low-return regime occupies about `24` months, or roughly `11.1%` of the sample

The macro averages also line up with the economic story we want:

- in the low-return regime, `VIX_z_exp` is about `1.486`, `CREDIT_z_exp` is about `1.098`, and `UNRATE_z_exp` is about `0.734`
- industrial production and activity are weaker there as well, with `INDPRO_YOY_z_exp` around `-1.212` and `CFNAI_z_exp` around `-1.144`

That combination makes this model the best centerpiece for the final project: it is dynamic, interpretable, and tied closely to recognizable stress conditions.


## 17. Allocation Rule and Backtest Design

Now we will connect the regime model to an allocation decision.

A very important detail:

- **smoothed** probabilities use future information and are good for interpretation
- **filtered** probabilities only use information available up to time `t`, so they are the right choice for a backtest

We will build a simple regime-aware strategy:

- when the model sees a higher probability of the low-return regime, shift toward a defensive portfolio
- when the low-return regime probability is low, stay closer to a risk-on portfolio

This is intentionally simple and interpretable. It gives us a clean first backtest before trying more advanced optimization.


In [ ]:
filtered_probs = ms_result.filtered_marginal_probabilities.iloc[:, ms_order].copy()
filtered_probs.columns = [f"ms_regime_{i}_filtered_prob" for i in range(2)]

backtest_df = pd.concat([ms_df[RETURN_COLS + ["equal_weight_return"]], filtered_probs], axis=1)
backtest_df.head()

In [ ]:
risk_on_weights = pd.Series(
    {
        "SPY_ret": 0.30,
        "EFA_ret": 0.20,
        "TLT_ret": 0.10,
        "GLD_ret": 0.10,
        "DBC_ret": 0.15,
        "VNQ_ret": 0.15,
    }
)

risk_off_weights = pd.Series(
    {
        "SPY_ret": 0.05,
        "EFA_ret": 0.05,
        "TLT_ret": 0.45,
        "GLD_ret": 0.25,
        "DBC_ret": 0.05,
        "VNQ_ret": 0.15,
    }
)

pd.DataFrame({"risk_on": risk_on_weights, "risk_off": risk_off_weights})

In [ ]:
dynamic_weights = pd.DataFrame(index=backtest_df.index, columns=RETURN_COLS, dtype=float)

for date, p_low in backtest_df["ms_regime_0_filtered_prob"].items():
    dynamic_weights.loc[date] = p_low * risk_off_weights + (1 - p_low) * risk_on_weights

dynamic_weights.head()

In [ ]:
strategy_returns = pd.DataFrame(index=backtest_df.index)
strategy_returns["regime_dynamic"] = (dynamic_weights.shift(1) * backtest_df[RETURN_COLS]).sum(axis=1)
strategy_returns["equal_weight"] = backtest_df[RETURN_COLS].mean(axis=1)
strategy_returns["sixty_forty"] = 0.60 * backtest_df["SPY_ret"] + 0.40 * backtest_df["TLT_ret"]
strategy_returns = strategy_returns.dropna()

strategy_returns.head()

In [ ]:
def performance_metrics(returns: pd.Series) -> pd.Series:
    wealth = (1 + returns).cumprod()
    drawdown = wealth / wealth.cummax() - 1
    ann_return = wealth.iloc[-1] ** (12 / len(returns)) - 1
    ann_vol = returns.std() * np.sqrt(12)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
    return pd.Series(
        {
            "annualized_return": ann_return,
            "annualized_volatility": ann_vol,
            "sharpe_ratio": sharpe,
            "max_drawdown": drawdown.min(),
            "cumulative_return": wealth.iloc[-1] - 1,
        }
    )


metrics_table = strategy_returns.apply(performance_metrics).T

turnover = dynamic_weights.shift(1).sub(dynamic_weights.shift(2)).abs().sum(axis=1).dropna()
metrics_table["avg_monthly_turnover"] = np.nan
metrics_table.loc["regime_dynamic", "avg_monthly_turnover"] = turnover.mean()

metrics_table.round(4)

In [ ]:
cumulative_backtest = (1 + strategy_returns).cumprod()

ax = cumulative_backtest.plot(figsize=(12, 6), linewidth=2)
ax.set_title("Backtest: Regime-Aware Allocation vs Benchmarks")
ax.set_xlabel("Date")
ax.set_ylabel("Growth of $1")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
ax = dynamic_weights.plot(figsize=(12, 6), linewidth=1.8)
ax.set_title("Dynamic Portfolio Weights Over Time")
ax.set_xlabel("Date")
ax.set_ylabel("Weight")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(backtest_df.index, backtest_df["ms_regime_0_filtered_prob"], linewidth=1.8, color="tab:red")
axes[0].set_title("Filtered Probability of Low-Return Regime")
axes[0].set_ylabel("Probability")
axes[0].grid(alpha=0.3)

axes[1].plot(turnover.index, turnover, linewidth=1.5, color="tab:purple")
axes[1].set_title("Strategy Turnover")
axes[1].set_ylabel("Turnover")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
backtest_output = pd.concat([backtest_df, dynamic_weights.add_prefix("w_"), strategy_returns], axis=1)
backtest_output_path = ROOT / "regime_backtest_results.csv"
backtest_output.to_csv(backtest_output_path)
print("Saved:", backtest_output_path)

## 18. Backtest Interpretation

The first regime-aware backtest shows that the regime signal can be turned into a workable trading rule.

On the aligned comparison sample:

- the template regime strategy earns about `5.93%` annualized return with volatility about `10.56%`
- its Sharpe ratio is about `0.5618`
- its max drawdown is about `-22.63%`
- average monthly turnover is about `0.0506`

Relative to the benchmarks, the template strategy is best viewed as a useful first proof of concept:

- it is slightly weaker than equal-weight on annualized return (`5.93%` versus `6.08%`) and Sharpe ratio (`0.5618` versus `0.6029`)
- it has worse drawdown than equal-weight (`-22.63%` versus `-19.59%`)
- it does, however, improve drawdown relative to the `60/40` benchmark (`-22.63%` versus `-27.51%`)
- most importantly, it remains very easy to explain because its weights come directly from the filtered low-regime probability

The main takeaway is that the regime signal is economically meaningful, but the first allocation rule is still too simple. That is why the refined allocation layer matters.


## 19. Refined Allocation Layer

The template strategy above is useful because it is easy to understand, but it still uses hand-set portfolio weights.

We can improve on that by estimating regime-specific portfolios directly from the data.

The idea is:

- use the filtered regime probabilities to weight past observations
- estimate a separate historical mean and covariance structure for the low-return and high-return regimes
- compute a long-only mean-variance style portfolio for each regime
- blend those two regime portfolios using the current filtered low-regime probability

To keep the optimizer stable and interpretable, we also apply:

- an expanding estimation window
- long-only weights
- a weight cap and a small minimum weight floor
- a blend with equal-weight to avoid extreme concentration


In [ ]:
def project_weights(raw_weights, cap=0.45, floor=0.03):
    """Project raw weights to a long-only, capped, fully-invested portfolio."""
    weights = np.clip(np.asarray(raw_weights, dtype=float), 0, None)
    if weights.sum() <= 1e-12:
        weights = np.ones_like(weights)
    weights = weights / weights.sum()

    for _ in range(20):
        weights = np.maximum(weights, floor)
        over_cap = weights > cap
        if not over_cap.any():
            break

        excess = (weights[over_cap] - cap).sum()
        weights[over_cap] = cap
        under_cap = ~over_cap

        if under_cap.sum() == 0:
            break

        allocation = weights[under_cap]
        if allocation.sum() <= 1e-12:
            allocation = np.ones(under_cap.sum()) / under_cap.sum()
        else:
            allocation = allocation / allocation.sum()

        weights[under_cap] += excess * allocation
        weights = weights / weights.sum()

    return weights / weights.sum()


def weighted_mean_cov(returns_matrix, regime_weights):
    """Weighted historical mean and covariance using regime probabilities."""
    weights = np.clip(np.asarray(regime_weights, dtype=float), 0, None)
    if weights.sum() <= 1e-12:
        weights = np.ones_like(weights) / len(weights)
    else:
        weights = weights / weights.sum()

    mean_vec = (returns_matrix * weights[:, None]).sum(axis=0)
    centered = returns_matrix - mean_vec
    cov_mat = (centered * weights[:, None]).T @ centered
    return mean_vec, cov_mat


def long_only_mean_variance(mean_vec, cov_mat, ridge=2e-4):
    """Simple stabilized mean-variance score before projection to portfolio weights."""
    cov_reg = cov_mat + ridge * np.eye(len(mean_vec))
    try:
        raw = np.linalg.solve(cov_reg, mean_vec)
    except np.linalg.LinAlgError:
        raw = np.linalg.pinv(cov_reg) @ mean_vec

    if np.all(raw <= 0):
        raw = 1 / np.sqrt(np.clip(np.diag(cov_reg), 1e-8, None))

    return raw


In [ ]:
lookback_months = 36
optimizer_blend = 0.65
base_equal_weight = np.repeat(1 / len(RETURN_COLS), len(RETURN_COLS))

refined_weights = []
refined_index = []

for t in range(lookback_months, len(backtest_df)):
    history = backtest_df.iloc[:t]
    history_returns = history[RETURN_COLS].to_numpy()

    low_mean, low_cov = weighted_mean_cov(history_returns, history["ms_regime_0_filtered_prob"].to_numpy())
    high_mean, high_cov = weighted_mean_cov(history_returns, history["ms_regime_1_filtered_prob"].to_numpy())

    low_regime_port = project_weights(
        optimizer_blend * project_weights(long_only_mean_variance(low_mean, low_cov))
        + (1 - optimizer_blend) * base_equal_weight
    )
    high_regime_port = project_weights(
        optimizer_blend * project_weights(long_only_mean_variance(high_mean, high_cov))
        + (1 - optimizer_blend) * base_equal_weight
    )

    p_low_now = backtest_df.iloc[t]["ms_regime_0_filtered_prob"]
    combined_weights = project_weights(p_low_now * low_regime_port + (1 - p_low_now) * high_regime_port)

    refined_weights.append(combined_weights)
    refined_index.append(backtest_df.index[t])

refined_weights = pd.DataFrame(refined_weights, index=refined_index, columns=RETURN_COLS)
refined_weights.head()


In [ ]:
refined_returns = pd.DataFrame(index=refined_weights.index)
refined_returns["template_regime_dynamic"] = (
    dynamic_weights.loc[refined_weights.index].shift(1) * backtest_df.loc[refined_weights.index, RETURN_COLS]
).sum(axis=1)
refined_returns["refined_regime_dynamic"] = (
    refined_weights.shift(1) * backtest_df.loc[refined_weights.index, RETURN_COLS]
).sum(axis=1)
refined_returns["equal_weight"] = backtest_df.loc[refined_weights.index, RETURN_COLS].mean(axis=1)
refined_returns["sixty_forty"] = (
    0.60 * backtest_df.loc[refined_weights.index, "SPY_ret"]
    + 0.40 * backtest_df.loc[refined_weights.index, "TLT_ret"]
)
refined_returns = refined_returns.dropna()

refined_metrics = refined_returns.apply(performance_metrics).T
refined_metrics["avg_monthly_turnover"] = np.nan
refined_metrics.loc[
    "template_regime_dynamic", "avg_monthly_turnover"
] = dynamic_weights.loc[refined_weights.index].diff().abs().sum(axis=1).dropna().mean()
refined_metrics.loc[
    "refined_regime_dynamic", "avg_monthly_turnover"
] = refined_weights.diff().abs().sum(axis=1).dropna().mean()

refined_metrics.round(4)


In [ ]:
refined_cumulative = (1 + refined_returns).cumprod()

ax = refined_cumulative.plot(figsize=(12, 6), linewidth=2)
ax.set_title("Refined Allocation Comparison")
ax.set_xlabel("Date")
ax.set_ylabel("Growth of $1")
ax.grid(alpha=0.3)
plt.show()


In [ ]:
ax = refined_weights.plot(figsize=(12, 6), linewidth=1.8)
ax.set_title("Refined Data-Driven Portfolio Weights")
ax.set_xlabel("Date")
ax.set_ylabel("Weight")
ax.grid(alpha=0.3)
plt.show()


## 20. Refined Allocation Interpretation

The refined allocation layer is the strongest allocation result in the notebook because it replaces hand-set regime portfolios with portfolios estimated from the data.

Compared with the template version, the refined strategy improves several headline metrics:

- annualized return rises from about `5.93%` to about `7.51%`
- annualized volatility falls slightly from about `10.56%` to about `10.36%`
- the Sharpe ratio improves from about `0.5618` to about `0.7243`
- average monthly turnover also falls slightly from about `0.0506` to about `0.0463`

The main cost is that max drawdown becomes a bit worse than the template strategy (`-23.74%` versus `-22.63%`) and also worse than equal-weight (`-19.59%`).

That tradeoff is important for the conclusion: regime information appears genuinely useful, but how we convert that information into portfolio weights materially affects the final performance profile.


In [ ]:
refined_backtest_output = pd.concat(
    [
        backtest_df.loc[refined_weights.index],
        refined_weights.add_prefix("refined_w_"),
        refined_returns,
    ],
    axis=1,
)
refined_backtest_output_path = ROOT / "refined_regime_backtest_results.csv"
refined_backtest_output.to_csv(refined_backtest_output_path)
print("Saved:", refined_backtest_output_path)


## 21. Conclusion

This notebook now tells a full project story from data construction to regime-aware allocation.

Final conclusions:

- The exploratory analysis supports a regime-based view of markets. Asset returns move in clusters, volatility changes over time, and macro stress indicators line up with recognizable episodes of weak performance.
- The Bayesian HMM provides the project with a real Bayesian machine learning component. It identifies a low-return, high-volatility state with persistent transitions and sensible posterior uncertainty.
- The two-state Markov-switching regression is the most stable and interpretable dynamic regime model in the notebook, so it should be the main empirical model in the final presentation and report.
- The allocation experiments show that regime probabilities are economically useful. The refined data-driven regime strategy improves meaningfully over the hand-built template strategy and also improves on equal-weight in annualized return and Sharpe ratio.
- At the same time, a simple `60/40` benchmark remains very strong in this sample. That is an honest and important result: good regime detection does not automatically guarantee the best portfolio unless the allocation rule is also well designed.

The best way to present the final project is therefore not to claim that regime modeling solves asset allocation by itself. A stronger conclusion is this:

> Bayesian and regime-switching models can identify economically meaningful market states, and those states can improve dynamic allocation decisions, but the quality of the final portfolio still depends heavily on how the regime signal is translated into weights.

That is a realistic, technically sound, and defensible conclusion for a Bayesian Machine Learning course project.
